In [1]:
# ------------------------------------------------------------
# LOAD df_bookings + icao_cluster.csv (FIXED)
# ------------------------------------------------------------

import pandas as pd
import gcsfs
import zipfile

# 1. Load df_bookings from ZIP (WORKING)
url_zip = "gs://agntworks-data-dev/wheelsup/raw/AgntWorks.zip"
fs = gcsfs.GCSFileSystem()

with fs.open(url_zip.replace('gs://', '')) as f:
    zf = zipfile.ZipFile(f)
    df_bookings = pd.read_csv(zf.open('AgntWorks/booked_flights.csv'), low_memory=False)

# 2. Load icao_cluster.csv via GCS (SAME AUTH)
url_icao = "gs://agntworks-data-dev/sandbox/experiments/icao_cluster.csv"
df_icao = pd.read_csv(fs.open(url_icao.replace('gs://', '')))

print(f"✅ df_bookings: {df_bookings.shape}")
print(f"✅ df_icao:    {df_icao.shape}")
print(f"\nMemory: {df_bookings.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
print("\ndf_bookings preview:")
print(df_bookings.head(2))
print("\ndf_icao preview:")
print(df_icao.head(2))

/opt/conda/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.cloud.storage_control_v2 once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.cloud.storage_control_v2 past that date.
  warnings.warn(message, FutureWarning)


✅ df_bookings: (258370, 43)
✅ df_icao:    (38859, 2)

Memory: 447.9 MB

df_bookings preview:
   flightId  atlasreservationid flightStatus  legOrder  flightoriginAirportId  \
0   1208327              794518    Cancelled         1                    810   
1   1208330              794521        Flown         1                    333   

  flightoriginAirport   flightoriginAirportName flightoriginAirportCity  \
0                KTEB                 Teterboro               Teterboro   
1                KPBI  Palm Beach International         West Palm Beach   

  flightoriginAirportState flightoriginAirportCountry  ...  \
0                       NJ                         US  ...   
1                       FL                         US  ...   

   flightactualAircraftTypeName             operatorName operatorType  \
0                           NaN         Pending Operator         TECH   
1                        CE-750  Wheels Up Partners, LLC        ONE_P   

  flightcost reservationId res

In [2]:
# ------------------------------------------------------------
# ADD 3 CLUSTER FEATURES to df_bookings
# ------------------------------------------------------------

# Create mapping dictionaries
icao_to_cluster = dict(zip(df_icao['icao'], df_icao['cluster']))

# 1. from_cluster (origin airport → cluster)
df_bookings['from_cluster'] = df_bookings['flightoriginAirport'].map(icao_to_cluster)

# 2. to_cluster (destination airport → cluster)  
df_bookings['to_cluster'] = df_bookings['flightdestinationAirport'].map(icao_to_cluster)

# 3. corridor (from_cluster - to_cluster)
df_bookings['corridor'] = df_bookings['from_cluster'] + ' - ' + df_bookings['to_cluster']

print("✅ 3 New Features Added!")
print(f"  • from_cluster: {df_bookings['from_cluster'].notna().sum():,} mapped ({df_bookings['from_cluster'].notna().mean()*100:.1f}%)")
print(f"  • to_cluster:   {df_bookings['to_cluster'].notna().sum():,} mapped ({df_bookings['to_cluster'].notna().mean()*100:.1f}%)")
print(f"  • corridor:     {df_bookings['corridor'].notna().sum():,} created")

# Preview
print("\nSample corridors:")
print(df_bookings[['flightoriginAirport', 'from_cluster', 
                   'flightdestinationAirport', 'to_cluster', 'corridor']].head())

✅ 3 New Features Added!
  • from_cluster: 255,431 mapped (98.9%)
  • to_cluster:   255,447 mapped (98.9%)
  • corridor:     252,611 created

Sample corridors:
  flightoriginAirport           from_cluster flightdestinationAirport  \
0                KTEB       NEW_YORK_CLUSTER                     KVRB   
1                KPBI          MIAMI_CLUSTER                     KTEB   
2                KBKL  WASHINGTON_DC_CLUSTER                     KHPN   
3                KYNG  WASHINGTON_DC_CLUSTER                     KROC   
4                KHXD        ATLANTA_CLUSTER                     KASH   

         to_cluster                                  corridor  
0   ORLANDO_CLUSTER        NEW_YORK_CLUSTER - ORLANDO_CLUSTER  
1  NEW_YORK_CLUSTER          MIAMI_CLUSTER - NEW_YORK_CLUSTER  
2  NEW_YORK_CLUSTER  WASHINGTON_DC_CLUSTER - NEW_YORK_CLUSTER  
3  NEW_YORK_CLUSTER  WASHINGTON_DC_CLUSTER - NEW_YORK_CLUSTER  
4    BOSTON_CLUSTER          ATLANTA_CLUSTER - BOSTON_CLUSTER  


In [3]:
# ------------------------------------------------------------
# ADD WHEELS UP SUBSIDIARIES FLAG
# ------------------------------------------------------------

# Create binary flag for Wheels Up subsidiaries
df_bookings['wu_subsidiaries'] = df_bookings['operatorName'].isin([
    'Wheels Up Private Jets, LLC',
    'Wheels Up Partners, LLC'
]).astype(int)

print("✅ 'wu_subsidiaries' flag added!")
print(f"  • Wheels Up rows:   {df_bookings['wu_subsidiaries'].sum():,} (True)")
print(f"  • Non-Wheels Up:    {len(df_bookings) - df_bookings['wu_subsidiaries'].sum():,} (False)")
print(f"  • Coverage:         {df_bookings['wu_subsidiaries'].mean()*100:.1f}%")

# Quick validation
print("\nSample with flag:")
print(df_bookings[['operatorName', 'wu_subsidiaries', 'from_cluster', 'to_cluster', 'corridor']].head())

✅ 'wu_subsidiaries' flag added!
  • Wheels Up rows:   179,316 (True)
  • Non-Wheels Up:    79,054 (False)
  • Coverage:         69.4%

Sample with flag:
                  operatorName  wu_subsidiaries           from_cluster  \
0             Pending Operator                0       NEW_YORK_CLUSTER   
1      Wheels Up Partners, LLC                1          MIAMI_CLUSTER   
2       Circadian Aviation LLC                0  WASHINGTON_DC_CLUSTER   
3       Circadian Aviation LLC                0  WASHINGTON_DC_CLUSTER   
4  Wheels Up Private Jets, LLC                1        ATLANTA_CLUSTER   

         to_cluster                                  corridor  
0   ORLANDO_CLUSTER        NEW_YORK_CLUSTER - ORLANDO_CLUSTER  
1  NEW_YORK_CLUSTER          MIAMI_CLUSTER - NEW_YORK_CLUSTER  
2  NEW_YORK_CLUSTER  WASHINGTON_DC_CLUSTER - NEW_YORK_CLUSTER  
3  NEW_YORK_CLUSTER  WASHINGTON_DC_CLUSTER - NEW_YORK_CLUSTER  
4    BOSTON_CLUSTER          ATLANTA_CLUSTER - BOSTON_CLUSTER  


In [4]:
# ------------------------------------------------------------
# operatorType breakdown: WU vs NON-WU
# ------------------------------------------------------------

print("OPERATOR TYPE BY WHEELS UP STATUS")
print("="*50)

# 1. WHEELS UP SUBSIDIARIES (True)
wu_operators = df_bookings[df_bookings['wu_subsidiaries'] == 1]['operatorType'].value_counts()
print("✅ WU_SUBSIDIARIES = TRUE:")
print(wu_operators)
print(f"  Total WU rows: {wu_operators.sum():,}")

print("\n" + "-"*50)

# 2. NON-WHEELS UP (False)  
non_wu_operators = df_bookings[df_bookings['wu_subsidiaries'] == 0]['operatorType'].value_counts()
print("❌ WU_SUBSIDIARIES = FALSE:")
print(non_wu_operators)
print(f"  Total NON-WU rows: {non_wu_operators.sum():,}")

print("\n" + "="*50)
print("💡 INTERPRETATION:")
print("ONE_P   = Wheels Up 'in-house' flights")
print("THREE_P = 3rd party operators") 
print("TECH    = Tech platform bookings")

OPERATOR TYPE BY WHEELS UP STATUS
✅ WU_SUBSIDIARIES = TRUE:
operatorType
ONE_P    179316
Name: count, dtype: int64
  Total WU rows: 179,316

--------------------------------------------------
❌ WU_SUBSIDIARIES = FALSE:
operatorType
THREE_P    56490
TECH       19804
ONE_P       2392
Name: count, dtype: int64
  Total NON-WU rows: 78,686

💡 INTERPRETATION:
ONE_P   = Wheels Up 'in-house' flights
THREE_P = 3rd party operators
TECH    = Tech platform bookings


In [5]:
# ------------------------------------------------------------
# TOP 30 CORRIDORS by REVENUE - WU + Non-WU (Full Marketplace)
# ------------------------------------------------------------

# Filter Flown flights only (revenue-generating)
flown_data = df_bookings[
    (df_bookings['flightStatus'] == 'Flown') & 
    (df_bookings['flightcost'].notna()) &
    (df_bookings['corridor'].notna())
].copy()

print("🏆 TOP 30 CORRIDORS - FULL WHEELS UP MARKETPLACE")
print("="*80)

# 1. WHEELS UP SUBSIDIARIES (Core Fleet)
wu_corridors = flown_data[flown_data['wu_subsidiaries'] == 1]
wu_top30 = wu_corridors.groupby('corridor')['flightcost'].sum().sort_values(ascending=False).head(30)

print("\n✅ WHEELS UP CORE FLEET (wu_subsidiaries=1)")
print(f"Total Revenue: ${wu_top30.sum():,.0f}")
print(wu_top30.round(0).astype(int).head(10) / 1e6)  # Top 10 in $M

print("\n" + "-"*80)

# 2. FULL MARKETPLACE (WU + 3rd Party)
marketplace_corridors = flown_data[flown_data['wu_subsidiaries'] == 0]
marketplace_top30 = marketplace_corridors.groupby('corridor')['flightcost'].sum().sort_values(ascending=False).head(30)

print("🌐 WHEELS UP MARKETPLACE (3rd Party + Overflow)")
print(f"Total Revenue: ${marketplace_top30.sum():,.0f}")
print(marketplace_top30.round(0).astype(int).head(10) / 1e6)  # Top 10 in $M

print("\n" + "="*80)
print("💰 COMBINED TOP 10 CORRIDORS (Full Marketplace)")
combined_top30 = flown_data.groupby('corridor')['flightcost'].sum().sort_values(ascending=False).head(10)
print(combined_top30.round(0).astype(int) / 1e6)  # $M

🏆 TOP 30 CORRIDORS - FULL WHEELS UP MARKETPLACE

✅ WHEELS UP CORE FLEET (wu_subsidiaries=1)
Total Revenue: $942,547,384
corridor
ATLANTA_CLUSTER - ATLANTA_CLUSTER          75.248475
CHICAGO_CLUSTER - ATLANTA_CLUSTER          44.350363
ORLANDO_CLUSTER - ATLANTA_CLUSTER          43.610827
ATLANTA_CLUSTER - ORLANDO_CLUSTER          40.003360
CHICAGO_CLUSTER - MIAMI_CLUSTER            39.764000
MIAMI_CLUSTER - ATLANTA_CLUSTER            39.651832
ATLANTA_CLUSTER - CHICAGO_CLUSTER          39.282410
NEW_YORK_CLUSTER - ATLANTA_CLUSTER         36.386796
WASHINGTON_DC_CLUSTER - ATLANTA_CLUSTER    35.296548
MIAMI_CLUSTER - CHICAGO_CLUSTER            34.406641
Name: flightcost, dtype: float64

--------------------------------------------------------------------------------
🌐 WHEELS UP MARKETPLACE (3rd Party + Overflow)
Total Revenue: $352,532,235
corridor
CHICAGO_CLUSTER - MIAMI_CLUSTER          23.100003
NEW_YORK_CLUSTER - MIAMI_CLUSTER         20.152453
MIAMI_CLUSTER - CHICAGO_CLUSTER         

In [6]:
import pandas as pd

# 1. Filter for Flown flights within the 'Light' cabin category
# We use .copy() to avoid SettingWithCopy warnings during the dropna step
df_light_flown = df_bookings[
    (df_bookings['flightStatus'] == 'Flown') & 
    (df_bookings['flightactualAircraftCabinName'] == 'Light')
].copy()

# 2. Exclude rows where corridor is null
df_light_flown = df_light_flown.dropna(subset=['corridor'])

# 3. Aggregate Revenue by Corridor
top_20_corridors = (
    df_light_flown.groupby('corridor')['flightcost']
    .sum()
    .sort_values(ascending=False)
    .head(20)
    .reset_index()
)

# 4. Clean up column names for the output
top_20_corridors.rename(columns={'flightcost': 'revenue'}, inplace=True)

# Format the revenue for better readability (optional)
# top_20_corridors['revenue_formatted'] = top_20_corridors['revenue'].apply(lambda x: f"${x:,.2f}")

print("Top 20 Corridors by Revenue (Status: Flown | Cabin: Light)")
print(top_20_corridors)

Top 20 Corridors by Revenue (Status: Flown | Cabin: Light)
                                    corridor      revenue
0            MIAMI_CLUSTER - ATLANTA_CLUSTER  19783632.96
1          ATLANTA_CLUSTER - ATLANTA_CLUSTER  19287801.50
2          CHICAGO_CLUSTER - ATLANTA_CLUSTER  17088888.42
3          ORLANDO_CLUSTER - ATLANTA_CLUSTER  15399978.91
4         NEW_YORK_CLUSTER - ATLANTA_CLUSTER  14837158.76
5            ATLANTA_CLUSTER - MIAMI_CLUSTER  14729844.20
6          ATLANTA_CLUSTER - CHICAGO_CLUSTER  13939638.30
7          ATLANTA_CLUSTER - ORLANDO_CLUSTER  13726640.29
8    WASHINGTON_DC_CLUSTER - ATLANTA_CLUSTER  13652132.15
9    ATLANTA_CLUSTER - WASHINGTON_DC_CLUSTER  12394433.52
10        ATLANTA_CLUSTER - NEW_YORK_CLUSTER  11489698.54
11         CHICAGO_CLUSTER - CHICAGO_CLUSTER   8882978.05
12           CHICAGO_CLUSTER - MIAMI_CLUSTER   8791400.81
13     MIAMI_CLUSTER - WASHINGTON_DC_CLUSTER   8661402.78
14  NEW_YORK_CLUSTER - WASHINGTON_DC_CLUSTER   7500932.63
15           

In [7]:
import pandas as pd

# 1. Filter for Flown flights in the 'Light' cabin
# We also add the condition to ensure flightcost is greater than 0
df_clean_revenue = df_bookings[
    (df_bookings['flightStatus'] == 'Flown') & 
    (df_bookings['flightactualAircraftCabinName'] == 'Light') &
    (df_bookings['flightcost'] > 0)
].copy()

# 2. Exclude rows where the corridor is null
# Using .dropna() ensures we don't have 'None - None' clusters
df_clean_revenue = df_clean_revenue.dropna(subset=['corridor'])

# 3. Group by corridor and sum the revenue
top_20_corridors = (
    df_clean_revenue.groupby('corridor')['flightcost']
    .sum()
    .sort_values(ascending=False)
    .head(20)
    .reset_index()
)

# 4. Final formatting
top_20_corridors.rename(columns={'flightcost': 'revenue'}, inplace=True)

print("Top 20 Corridors by Revenue (Status: Flown | Cabin: Light | Revenue > 0)")
print(top_20_corridors)

Top 20 Corridors by Revenue (Status: Flown | Cabin: Light | Revenue > 0)
                                    corridor      revenue
0            MIAMI_CLUSTER - ATLANTA_CLUSTER  19783632.96
1          ATLANTA_CLUSTER - ATLANTA_CLUSTER  19287801.50
2          CHICAGO_CLUSTER - ATLANTA_CLUSTER  17088888.42
3          ORLANDO_CLUSTER - ATLANTA_CLUSTER  15399978.91
4         NEW_YORK_CLUSTER - ATLANTA_CLUSTER  14837158.76
5            ATLANTA_CLUSTER - MIAMI_CLUSTER  14729844.20
6          ATLANTA_CLUSTER - CHICAGO_CLUSTER  13939638.30
7          ATLANTA_CLUSTER - ORLANDO_CLUSTER  13726640.29
8    WASHINGTON_DC_CLUSTER - ATLANTA_CLUSTER  13652132.15
9    ATLANTA_CLUSTER - WASHINGTON_DC_CLUSTER  12394433.52
10        ATLANTA_CLUSTER - NEW_YORK_CLUSTER  11489698.54
11         CHICAGO_CLUSTER - CHICAGO_CLUSTER   8882978.05
12           CHICAGO_CLUSTER - MIAMI_CLUSTER   8791400.81
13     MIAMI_CLUSTER - WASHINGTON_DC_CLUSTER   8661402.78
14  NEW_YORK_CLUSTER - WASHINGTON_DC_CLUSTER   7500932.63

In [8]:
# List of specific models as they appear in your flightactualAircraftTypeName
target_models = ['Phenom 300', 'CL-300', 'Phenom 300 (G3000)']

# Apply the same logic as before to maintain consistency
df_models_check = df_bookings[
    (df_bookings['flightStatus'] == 'Flown') & 
    (df_bookings['flightactualAircraftTypeName'].isin(target_models)) &
    (df_bookings['flightcost'] > 0)
].copy()

# Get unique cabin categories for these specific models
cabin_mapping = df_models_check.groupby(['flightactualAircraftTypeName', 'flightactualAircraftCabinName']).size().reset_index(name='count')

print("Cabin Classification for Target Models:")
print(cabin_mapping)

Cabin Classification for Target Models:
  flightactualAircraftTypeName flightactualAircraftCabinName  count
0                       CL-300             Premium Super-Mid   5225
1                       CL-300                 Super Midsize    222
2                   Phenom 300                         Light   2265
3                   Phenom 300                 Premium Light  23655
4           Phenom 300 (G3000)                 Premium Light   1143


In [9]:
import pandas as pd

# 1. Define the Cabin Names that represent your target TAM
# This covers both the legacy and the new 'Premium' tiers
target_cabins = [
    'Light', 
    'Premium Light', 
    'Super Midsize', 
    'Premium Super-Mid'
]

# 2. Filter the dataframe
df_segment_analysis = df_bookings[
    (df_bookings['flightStatus'] == 'Flown') & 
    (df_bookings['flightactualAircraftCabinName'].isin(target_cabins)) &
    (df_bookings['flightcost'] > 0)
].copy()

# 3. Clean null corridors
df_segment_analysis = df_segment_analysis.dropna(subset=['corridor'])

# 4. Group by corridor and cabin to see the revenue distribution
# We sum cost and also get the count to see the 'volume' vs 'revenue'
corridor_summary = df_segment_analysis.groupby(['corridor', 'flightactualAircraftCabinName']).agg(
    total_revenue=('flightcost', 'sum'),
    flight_count=('flightcost', 'count')
).reset_index()

# 5. Get the Top 20 Corridors by Total Revenue
top_20_revenue = (
    df_segment_analysis.groupby('corridor')['flightcost']
    .sum()
    .sort_values(ascending=False)
    .head(20)
    .reset_index()
)

print("Top 20 Corridors for Light/Super-Mid TAM (Legacy + Premium):")
print(top_20_revenue)

Top 20 Corridors for Light/Super-Mid TAM (Legacy + Premium):
                                   corridor   flightcost
0           CHICAGO_CLUSTER - MIAMI_CLUSTER  41382943.24
1           MIAMI_CLUSTER - CHICAGO_CLUSTER  38533740.09
2          NEW_YORK_CLUSTER - MIAMI_CLUSTER  36126742.03
3        NEW_YORK_CLUSTER - ATLANTA_CLUSTER  33816531.48
4           MIAMI_CLUSTER - ATLANTA_CLUSTER  33778190.06
5         CHICAGO_CLUSTER - ATLANTA_CLUSTER  32177550.44
6         ATLANTA_CLUSTER - ATLANTA_CLUSTER  31010910.00
7          MIAMI_CLUSTER - NEW_YORK_CLUSTER  29710484.20
8           ATLANTA_CLUSTER - MIAMI_CLUSTER  29048873.35
9         ATLANTA_CLUSTER - CHICAGO_CLUSTER  27634191.87
10       ATLANTA_CLUSTER - NEW_YORK_CLUSTER  27246686.99
11        ORLANDO_CLUSTER - ATLANTA_CLUSTER  26309218.45
12        CHICAGO_CLUSTER - ORLANDO_CLUSTER  24665144.34
13        ATLANTA_CLUSTER - ORLANDO_CLUSTER  24182481.13
14    WASHINGTON_DC_CLUSTER - MIAMI_CLUSTER  23813918.83
15        ORLANDO_CLUSTER -

In [10]:
import pandas as pd

# 1. Convert the departure time to datetime objects
# errors='coerce' will turn any garbled data into NaT (Not a Time)
df_bookings['flightActualDepartureTime'] = pd.to_datetime(df_bookings['flightActualDepartureTime'], errors='coerce')

# 2. Calculate the range
data_start = df_bookings['flightActualDepartureTime'].min()
data_end = df_bookings['flightActualDepartureTime'].max()

# 3. Calculate total duration for context
total_days = (data_end - data_start).days

print(f"--- Dataset Time Range ---")
print(f"Earliest Flight: {data_start}")
print(f"Latest Flight:   {data_end}")
print(f"Total Span:      {total_days} days (approx {total_days/365:.2f} years)")

--- Dataset Time Range ---
Earliest Flight: 2024-04-01 15:19:00+00:00
Latest Flight:   2026-04-23 15:20:00+00:00
Total Span:      752 days (approx 2.06 years)


In [11]:
import pandas as pd

# 1. Convert to Eastern Time
# First ensure it's datetime, then localize/convert to US/Eastern
# 'US/Eastern' automatically handles the switch between EST and EDT
if df_bookings['flightActualDepartureTime'].dt.tz is None:
    # If naive (no timezone), assume UTC then convert
    df_bookings['flightActualDepartureTime_EST'] = df_bookings['flightActualDepartureTime'].dt.tz_localize('UTC').dt.tz_convert('US/Eastern')
else:
    # If already has timezone info, just convert
    df_bookings['flightActualDepartureTime_EST'] = df_bookings['flightActualDepartureTime'].dt.tz_convert('US/Eastern')

# 2. Define our "Modernization TAM" filters
target_cabins = ['Light', 'Premium Light', 'Super Midsize', 'Premium Super-Mid']

# 3. Create the Base Filtered Set
# (Flown, Target Cabins, Revenue > 0, No Null Corridors)
df_clean_base = df_bookings[
    (df_bookings['flightStatus'] == 'Flown') &
    (df_bookings['flightactualAircraftCabinName'].isin(target_cabins)) &
    (df_bookings['flightcost'] > 0) &
    (df_bookings['corridor'].notna())
].copy()

# 4. Partition into the requested timeframes
# 2025 Full Year
df_2025_flown = df_clean_base[
    (df_clean_base['flightActualDepartureTime_EST'] >= '2025-01-01') & 
    (df_clean_base['flightActualDepartureTime_EST'] <= '2025-12-31 23:59:59')
].copy()

# 2026 Q1
df_q12026_flown = df_clean_base[
    (df_clean_base['flightActualDepartureTime_EST'] >= '2026-01-01') & 
    (df_clean_base['flightActualDepartureTime_EST'] <= '2026-03-31 23:59:59')
].copy()

print(f"✅ df_2025_flown created with {len(df_2025_flown):,} rows.")
print(f"✅ df_q12026_flown created with {len(df_q12026_flown):,} rows.")

✅ df_2025_flown created with 49,560 rows.
✅ df_q12026_flown created with 12,810 rows.


In [12]:
# 1. Identify the Top 20 Corridors by Revenue in 2025
top_20_corridors_list = (
    df_2025_flown.groupby('corridor')['flightcost']
    .sum()
    .sort_values(ascending=False)
    .head(20)
    .index
)

# 2. Filter 2025 data to only include these corridors
df_top_20_stats = df_2025_flown[df_2025_flown['corridor'].isin(top_20_corridors_list)]

# 3. Calculate Min, Max, Mean, and Standard Deviation
# Added 'std' because a high standard deviation tells you the 'Vibe' of the pricing is unpredictable
price_stats = df_top_20_stats.groupby('corridor')['flightcost'].agg(
    count='count',
    min_price='min',
    max_price='max',
    mean_price='mean',
    std_dev='std'
).reset_index()

# 4. Sort by the original revenue rank (which we pull from the top_20_corridors_list)
price_stats['corridor'] = pd.Categorical(price_stats['corridor'], categories=top_20_corridors_list, ordered=True)
price_stats = price_stats.sort_values('corridor')

# Format for readability
pd.options.display.float_format = '{:,.2f}'.format

print("--- 2025 Pricing Variation: Top 20 Revenue Corridors ---")
print(price_stats)

--- 2025 Pricing Variation: Top 20 Revenue Corridors ---
                                   corridor  count  min_price  max_price  \
7           CHICAGO_CLUSTER - MIAMI_CLUSTER    746  10,767.30  46,557.75   
13       NEW_YORK_CLUSTER - ATLANTA_CLUSTER    957       5.00  41,463.89   
10          MIAMI_CLUSTER - CHICAGO_CLUSTER    631  11,334.00  35,679.00   
9           MIAMI_CLUSTER - ATLANTA_CLUSTER    992   7,400.38  35,413.54   
6         CHICAGO_CLUSTER - ATLANTA_CLUSTER   1108     995.00  35,691.87   
2           ATLANTA_CLUSTER - MIAMI_CLUSTER    998     295.00  32,744.21   
15         NEW_YORK_CLUSTER - MIAMI_CLUSTER    526  15,120.00  47,983.33   
0         ATLANTA_CLUSTER - ATLANTA_CLUSTER   1409     495.00  32,985.30   
1         ATLANTA_CLUSTER - CHICAGO_CLUSTER    945   5,754.00  32,388.00   
3        ATLANTA_CLUSTER - NEW_YORK_CLUSTER    826       5.00  35,691.87   
16        ORLANDO_CLUSTER - ATLANTA_CLUSTER    975   4,425.24  31,485.00   
11         MIAMI_CLUSTER - NEW_

In [13]:
# 1. Filter for 'Flown' status only
df_flown = df_bookings[df_bookings['flightStatus'] == 'Flown'].copy()
total_flown = len(df_flown)

# 2. Define the specific fields you requested
fields_to_audit = [
    'flightId', 'atlasreservationid', 'flightStatus', 'legOrder',
    'flightoriginAirport', 'flightdestinationAirport', 
    'flightActualDepartureTime', 'flightActualArrivalTime', 
    'flightActualFlightHours', 'flightActualBilledHours',
    'flightactualAircraftCabinName', 'flightactualAircraftTypeName', 
    'operatorName', 'operatorType'
]

# 3. Calculate Null stats
null_stats = df_flown[fields_to_audit].isnull().sum()
null_percent = (null_stats / total_flown) * 100

# 4. Create and display the report
flown_audit_report = pd.DataFrame({
    'Null Count': null_stats,
    'Null %': null_percent
}).sort_values(by='Null %', ascending=False)

pd.options.display.float_format = '{:,.2f}%'.format
print(f"--- Operational Audit: Flown Flights Only (N={total_flown:,}) ---")
print(flown_audit_report)

--- Operational Audit: Flown Flights Only (N=220,957) ---
                               Null Count  Null %
flightActualArrivalTime             37210  16.84%
flightActualDepartureTime           35272  15.96%
flightActualBilledHours              6336   2.87%
flightActualFlightHours              4996   2.26%
flightactualAircraftTypeName          441   0.20%
operatorType                          349   0.16%
flightactualAircraftCabinName         176   0.08%
flightoriginAirport                     0   0.00%
flightdestinationAirport                0   0.00%
flightStatus                            0   0.00%
atlasreservationid                      0   0.00%
flightId                                0   0.00%
legOrder                                0   0.00%
operatorName                            0   0.00%


In [14]:
# 1. Identify the 'Time-Blind' flown flights
df_time_blind = df_bookings[
    (df_bookings['flightStatus'] == 'Flown') & 
    (df_bookings['flightActualDepartureTime'].isna())
]

# 2. Check the Distribution by Operator Type
print("--- Operator Type Breakdown for Time-Blind Flights ---")
print(df_time_blind['operatorType'].value_counts())

# 3. Check the Top 10 Specific Operators causing the gaps
print("\n--- Top 10 Operators with Missing Departure Times ---")
print(df_time_blind['operatorName'].value_counts().head(10))

# 4. CRITICAL: Calculate the 'Blindness Rate' per Operator Type
# This tells us: 'Of all flights by this operator type, what % are missing timestamps?'
total_flown_by_type = df_bookings[df_bookings['flightStatus'] == 'Flown']['operatorType'].value_counts()
blind_by_type = df_time_blind['operatorType'].value_counts()
blindness_rate = (blind_by_type / total_flown_by_type) * 100

print("\n--- Data Blindness Rate (%) by Operator Type ---")
print(blindness_rate.sort_values(ascending=False))

--- Operator Type Breakdown for Time-Blind Flights ---
operatorType
THREE_P    34803
ONE_P        182
TECH          28
Name: count, dtype: int64

--- Top 10 Operators with Missing Departure Times ---
operatorName
GrandView Aviation      5503
Baker Aviation          4240
Ventura Air Services    2592
Premier Private Jets    2382
Jet Linx Aviation       2174
Omni Air Transport      1532
Paragon Airways         1292
BellAir Aviation        1115
Fly Alliance             882
Ultimate Jetcharters     852
Name: count, dtype: int64

--- Data Blindness Rate (%) by Operator Type ---
operatorType
TECH      82.35%
THREE_P   65.06%
ONE_P      0.11%
Name: count, dtype: float64
